# Open Meteo API key for live data to train.

In [4]:
import numpy as np
import requests
import polars as pl
from polars import col
from datetime import datetime
import json
import pprint

pl.Config.set_tbl_cols(n = 15)
pl.Config.set_tbl_rows(n = 25)
pl.Config.set_tbl_width_chars(200)

polars.config.Config

# Testing Open Meteo API.

###                                            Testing to check what the API returns.

In [ ]:
# Prameters for the API request
params = {
    "latitude": 23.7106,        # for Dhaka. 
    "longitude": 90.4067,       # for Dhaka. 
    "start_date": "2024-01-01",
    "end_date"  : "2024-01-01",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m", # unit : %.
        "pressure_msl",         # unit : hPa. Sealevel pressure.
        "cloud_cover_low",      # unit : %. The most important cloud layer for rain.
        "cloud_cover",    # unit : %. Great overall context.
        "vapour_pressure_deficit", # unit : kPa (kilopascal).
        # "visibility",           # unit : meters. It's 'how far you can see clearly through the air' which changes frequently for
        "wind_speed_10m",         # very weather conditions INSTANT. Its a real-time sensor for CURRENT DATA, not for Historical.
        "wind_direction_10m",   # unit : °.
        "wind_gusts_10m",       # unit : km/h.
        "precipitation",              # precipitation = the amount of water that is expected to fall from the sky.
        # "precipitation_probability" # precipitation_probability = What's the chance it'll fall? That means its for FORECASTING,
    ],                                # not what already happened in the past i.e. Historical Data. Setting it will return NULLs.
    # "daily": "temperature_2m_max,temperature_2m_min,wind_speed_10m_max,wind_gusts_10m_max",
    "timezone": "Asia/Dhaka", # Set timezone to Dhaka to get 24 hours data aligned with Dhaka's Local time from 0 to 23 because
    # The data is for Dhaka, so timezone ofc need to be aligned with Dhaka's local time, not with another city/country's timezone.
    # units of the parameters whose units are not constant :
    "temperature_unit" : "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm"
}

# Make the API request.
url_temp = "https://archive-api.open-meteo.com/v1/archive"
response_temp = requests.get(url_temp, params = params)
data_temp = response_temp.json()

print(json.dumps(data_temp, indent = 4))

{
    "latitude": 23.725834,
    "longitude": 90.38015,
    "generationtime_ms": 0.6074905395507812,
    "utc_offset_seconds": 21600,
    "timezone": "Asia/Dhaka",
    "timezone_abbreviation": "GMT+6",
    "elevation": 27.0,
    "hourly_units": {
        "time": "iso8601",
        "temperature_2m": "\u00b0C",
        "relative_humidity_2m": "%",
        "pressure_msl": "hPa",
        "cloud_cover_low": "%",
        "cloud_cover": "%",
        "vapour_pressure_deficit": "kPa",
        "wind_speed_10m": "km/h",
        "wind_direction_10m": "\u00b0",
        "wind_gusts_10m": "km/h",
        "precipitation": "mm"
    },
    "hourly": {
        "time": [
            "2024-01-01T00:00",
            "2024-01-01T01:00",
            "2024-01-01T02:00",
            "2024-01-01T03:00",
            "2024-01-01T04:00",
            "2024-01-01T05:00",
            "2024-01-01T06:00",
            "2024-01-01T07:00",
            "2024-01-01T08:00",
            "2024-01-01T09:00",
            "2024-01

### Finally creating our Training Dataset (with feature engineering) from that API.

In [181]:
params = {
    "latitude": 23.7106,        # for Dhaka. 
    "longitude": 90.4067,       # for Dhaka. 
    "start_date": "2010-09-01",
    "end_date"  : "2025-09-01",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m", # unit : %.
        "pressure_msl",         # unit : hPa. Sealevel pressure.
        "cloud_cover_low",      # unit : %. The most important cloud layer for rain.
        "cloud_cover",    # unit : %. Great overall context.
        "vapour_pressure_deficit", # unit : kPa (kilopascal). VPD.
        # "visibility",           # unit : meters. It's 'how far you can see clearly through the air' which changes frequently for
        "wind_speed_10m",         # very weather conditions INSTANT. Its a real-time sensor for CURRENT DATA, not for Historical.
        "wind_direction_10m",   # unit : °.                         (alternative of 'visibility' are humidity, cloud_cover, VPD.)
        "wind_gusts_10m",       # unit : km/h.
        "precipitation",              # precipitation = the amount of water that is expected to fall from the sky.
        # "precipitation_probability" # precipitation_probability = What's the chance it'll fall? That means its for FORECASTING,
    ],                                # not what already happened in the past i.e. Historical Data. Setting it will return NULLs.
    # "daily": "temperature_2m_max,temperature_2m_min,wind_speed_10m_max,wind_gusts_10m_max",
    "timezone": "Asia/Dhaka", # Set timezone to Dhaka to get 24 hours data aligned with Dhaka's Local time from 0 to 23 because
    # The data is for Dhaka, so timezone ofc need to be aligned with Dhaka's local time, not with another city/country's timezone.
    # units of the parameters whose units are not constant :
    "temperature_unit" : "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm"
}

# Make the API request.
url = "https://archive-api.open-meteo.com/v1/archive"
response = requests.get(url, params = params)
data: dict = response.json()

# Extract hourly data.
hourly_data: dict = data.get("hourly", {}) # dict.get(key, default).
pprint.PrettyPrinter(width = 2).pprint(list(hourly_data.keys())) # Just to print vertically.

['time',
 'temperature_2m',
 'relative_humidity_2m',
 'pressure_msl',
 'cloud_cover_low',
 'cloud_cover',
 'vapour_pressure_deficit',
 'wind_speed_10m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'precipitation']


In [224]:
#                  Convert the dict/json data to dataframe with CORRECT TIME ZONE and Feature Engineering.

df = ( pl.DataFrame(data = hourly_data)
      .lazy()
      .rename({"time" : "date"}) # {old_name : new_name}.
      .with_columns(date  = col("date").str.to_datetime(format = "%Y-%m-%dT%H:%M", time_zone = data['timezone']))
      .with_columns(hour  = col('date').dt.hour(),  # 1 to 24 hours.
                    month = col('date').dt.month(), # 1 to 12 no month.
                    day_of_year = col('date').dt.ordinal_day()) # 1 to 366 days.
      .with_columns(rainfall = (  pl.when(col("precipitation") <= 0)
                                          .then(0)  # Clear Sky.
                                    .when(col("precipitation") < 2.1)
                                          .then(1)  # Light rain.
                                    .when(col("precipitation") < 10.1)
                                          .then(2)  # Noticeable rain.
                                    .when(col("precipitation") < 30.1)
                                          .then(3)  # Heavy rain.
                                    .when(col("precipitation") < 50.1)
                                          .then(4)  # Very heavy rain.
                                    .otherwise(5)  ).cast(pl.Int8)) # Extreme rainfall.
      .with_columns(col("relative_humidity_2m", "cloud_cover_low", "cloud_cover").cast(pl.Int8),
                    col("wind_direction_10m").cast(pl.Int16))
      .drop("date", "precipitation")
      .select(col("hour", "month", "day_of_year"),
              col('*').exclude("hour", "month", "day_of_year")) # Features at the front + Target column at the end.
      .collect()
)

print(df.head(5))
print("Number of rows and columns are =", df.shape)

shape: (5, 13)
┌──────┬───────┬─────────────┬────────────────┬────────────────────┬──────────────┬─────────────────┬─────────────┬───────────────────┬────────────────┬───────────────────┬────────────────┬──────────┐
│ hour ┆ month ┆ day_of_year ┆ temperature_2m ┆ relative_humidity_ ┆ pressure_msl ┆ cloud_cover_low ┆ cloud_cover ┆ vapour_pressure_d ┆ wind_speed_10m ┆ wind_direction_10 ┆ wind_gusts_10m ┆ rainfall │
│ ---  ┆ ---   ┆ ---         ┆ ---            ┆ 2m                 ┆ ---          ┆ ---             ┆ ---         ┆ eficit            ┆ ---            ┆ m                 ┆ ---            ┆ ---      │
│ i8   ┆ i8    ┆ i16         ┆ f64            ┆ ---                ┆ f64          ┆ i8              ┆ i8          ┆ ---               ┆ f64            ┆ ---               ┆ f64            ┆ i8       │
│      ┆       ┆             ┆                ┆ i8                 ┆              ┆                 ┆             ┆ f64               ┆                ┆ i16               ┆         

In [225]:
print(df.describe())

shape: (9, 14)
┌────────────┬──────────┬──────────┬─────────────┬───────────────┬───────────────┬──────────────┬───────────────┬─────────────┬───────────────┬───────────────┬──────────────┬──────────────┬──────────┐
│ statistic  ┆ hour     ┆ month    ┆ day_of_year ┆ temperature_2 ┆ relative_humi ┆ pressure_msl ┆ cloud_cover_l ┆ cloud_cover ┆ vapour_pressu ┆ wind_speed_10 ┆ wind_directi ┆ wind_gusts_1 ┆ rainfall │
│ ---        ┆ ---      ┆ ---      ┆ ---         ┆ m             ┆ dity_2m       ┆ ---          ┆ ow            ┆ ---         ┆ re_deficit    ┆ m             ┆ on_10m       ┆ 0m           ┆ ---      │
│ str        ┆ f64      ┆ f64      ┆ f64         ┆ ---           ┆ ---           ┆ f64          ┆ ---           ┆ f64         ┆ ---           ┆ ---           ┆ ---          ┆ ---          ┆ f64      │
│            ┆          ┆          ┆             ┆ f64           ┆ f64           ┆              ┆ f64           ┆             ┆ f64           ┆ f64           ┆ f64          ┆ f64   

In [226]:
# checking which row has at least 1 null value and which column(s).
print( df.filter(pl.any_horizontal(col('*').is_null()) == True) )
print(f"Number of rows before deleting null values = {df.shape[0]}.")
# dropping rows contain at least 1 null value.
df = df.drop_nulls()

# rechecking which row has at least 1 null value and which column(s).
print( df.filter(pl.any_horizontal(col('*').is_null()) == True) )
print(f"Number of rows after deleting null values = {df.shape[0]}.")

shape: (0, 13)
┌──────┬───────┬─────────────┬────────────────┬────────────────────┬──────────────┬─────────────────┬─────────────┬───────────────────┬────────────────┬───────────────────┬────────────────┬──────────┐
│ hour ┆ month ┆ day_of_year ┆ temperature_2m ┆ relative_humidity_ ┆ pressure_msl ┆ cloud_cover_low ┆ cloud_cover ┆ vapour_pressure_d ┆ wind_speed_10m ┆ wind_direction_10 ┆ wind_gusts_10m ┆ rainfall │
│ ---  ┆ ---   ┆ ---         ┆ ---            ┆ 2m                 ┆ ---          ┆ ---             ┆ ---         ┆ eficit            ┆ ---            ┆ m                 ┆ ---            ┆ ---      │
│ i8   ┆ i8    ┆ i16         ┆ f64            ┆ ---                ┆ f64          ┆ i8              ┆ i8          ┆ ---               ┆ f64            ┆ ---               ┆ f64            ┆ i8       │
│      ┆       ┆             ┆                ┆ i8                 ┆              ┆                 ┆             ┆ f64               ┆                ┆ i16               ┆         

In [227]:
# frequency per hour. There's total 8784 hours and among them 7000 hours were clear sky and ......
( df['rainfall']
 .value_counts(name = "count(hours)")
 .sort(by = "rainfall")
 .insert_column(index  = 2,
                column = pl.Series(name = 'rainfall_intensity',
                                   values = ["Clear sky", "Light rain", "Noticeable rain", "Heavy rain", "Very heavy rain"]))
)

rainfall,count(hours),rainfall_intensity
i8,u32,str
0,101754,"""Clear sky"""
1,26334,"""Light rain"""
2,3266,"""Noticeable rain"""
3,165,"""Heavy rain"""
4,1,"""Very heavy rain"""


# Bulding and running the Machine Learning Model.

```js
    Transformer like FunctionTransformer, CyclicalFeatures, they require Numpy array or Pandas dataframe, not polars dataframe. Thats why its better to convert the the Final Dataframe used for EDA, to Pandas dataframe to compatiable with everything.
```

In [259]:
df_pandas = df.to_pandas()
print(df_pandas.head(5))
print("\n Type =", type(df_pandas))

   hour  month  day_of_year  temperature_2m  relative_humidity_2m  \
0     0      9          243            26.5                    89   
1     1      9          243            26.5                    89   
2     2      9          243            26.2                    90   
3     3      9          243            26.0                    91   
4     4      9          243            26.0                    92   

   pressure_msl  cloud_cover_low  cloud_cover  vapour_pressure_deficit  \
0        1004.1               92           98                     0.37   
1        1003.9               78          100                     0.39   
2        1003.6               97          100                     0.33   
3        1003.2              100          100                     0.30   
4        1003.9               39           99                     0.27   

   wind_speed_10m  wind_direction_10m  wind_gusts_10m  rainfall  
0            16.4                 151            25.6         0  
1       

In [267]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, train_test_split, GridSearchCV
from feature_engine.creation import CyclicalFeatures
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
import joblib

### Testing if the preprocessor, column transformer is returning desired result.

In [ ]:
# def sin_transformer(period):
#     return FunctionTransformer(
#             func              = lambda degree: np.sin(degree * (2*np.pi / period)), # returns Radian Value.
#             feature_names_out = lambda transformer_name, features_names: [f"{f}_sin" for f in features_names])

# def cos_transformer(period):
#     return FunctionTransformer(
#             func              = lambda degree: np.cos(degree * (2*np.pi / period)), # returns Radian Value.
#             feature_names_out = lambda transformer_name, features_names: [f"{f}_cos" for f in features_names])

# preprocessor: ColumnTransformer = ColumnTransformer( # below we've performed cyclic_cossin_transformer only.
#     transformers = [ # transformer = (transformer_name, transformer_function, list_of_features_to_be_transformed).
#         ("hour_sin_transformer",        sin_transformer(24),   ["hour"]),
#         ("hour_cos_transformer",        cos_transformer(24),   ["hour"]),
#         ("month_sin_transformer",       sin_transformer(12),   ["month"]),
#         ("month_cos_transformer",       cos_transformer(12),   ["month"]),
#         ("day_of_year_sin_transformer", sin_transformer(366),  ["day_of_year"]),
#         ("day_of_year_cos_transformer", cos_transformer(366),  ["day_of_year"]),
#         ("wind_direction_10m_sin",      sin_transformer(360),  ["wind_direction_10m"]),
#         ("wind_direction_10m_cos",      cos_transformer(360),  ["wind_direction_10m"]),
#     ],
#     remainder = "passthrough", # remove the columns used above to trasform, keep the rest as they are after newly transformed 6 columns..
#     verbose_feature_names_out = False # Prefix the input features name (here it's e.g. hour_sin after function_transformer overridden 
# )                                     # the original input feature name) with their corresponding transformer_name or not.

# preprocessor.set_output(transform = "polars") # This is how we'll check what the preprocessor returning.

# final_df = preprocessor.fit_transform(df) # preprocessor will return polars dataframe AFTER this function is called.

# print(final_df.head(5), "\n\n")
# print(preprocessor.get_feature_names_out()) # The hour, month, day_of_year, wind_direction_10m columns auto dropped in ColumnTransformer.

shape: (5, 17)
┌──────────┬──────────┬───────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬───┬─────────────┬─────────────┬────────────┬────────────┬────────────┬────────────┬──────────┐
│ hour_sin ┆ hour_cos ┆ month_sin ┆ month_cos   ┆ day_of_year ┆ day_of_year ┆ wind_direct ┆ wind_direct ┆ … ┆ pressure_ms ┆ cloud_cover ┆ cloud_cove ┆ vapour_pre ┆ wind_speed ┆ wind_gusts ┆ rainfall │
│ ---      ┆ ---      ┆ ---       ┆ ---         ┆ _sin        ┆ _cos        ┆ ion_10m_sin ┆ ion_10m_cos ┆   ┆ l           ┆ _low        ┆ r          ┆ ssure_defi ┆ _10m       ┆ _10m       ┆ ---      │
│ f64      ┆ f64      ┆ f64       ┆ f64         ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆   ┆ ---         ┆ ---         ┆ ---        ┆ cit        ┆ ---        ┆ ---        ┆ i8       │
│          ┆          ┆           ┆             ┆ f64         ┆ f64         ┆ f64         ┆ f64         ┆   ┆ f64         ┆ i8          ┆ i8         ┆ ---        ┆ f64        ┆ f64 

In [260]:
cyclical_features = ["hour", "month", "day_of_year", "wind_direction_10m"]
cyclical_transformer = CyclicalFeatures(drop_original = True,
                                        max_values = {'hour':24, 'month':12, 'day_of_year':366, 'wind_direction_10m':360})

preprocessor: ColumnTransformer = ColumnTransformer( # below we've performed cyclic_cossin_transformer only.
    transformers = [("Cyclical_Transformer", cyclical_transformer, cyclical_features),], # (name, function, features_list).
    remainder = "passthrough", # remove the columns used above to trasform, keep the rest as they are after newly transformed 6 columns..
    verbose_feature_names_out = False # Prefix the input features name (here it's e.g. hour_sin after function_transformer overridden 
)                                     # the original input feature name) with their corresponding transformer_name or not.

preprocessor.set_output(transform = "polars") # This is how we'll check what the preprocessor returning.

final_df = preprocessor.fit_transform(df_pandas) # preprocessor will return polars dataframe AFTER this function is called.

print(final_df.head(5), "\n\n")
print(preprocessor.get_feature_names_out()) # The hour, month, day_of_year, wind_direction_10m columns auto dropped in ColumnTransformer.

shape: (5, 17)
┌──────────┬──────────┬───────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬───┬─────────────┬─────────────┬────────────┬────────────┬────────────┬────────────┬──────────┐
│ hour_sin ┆ hour_cos ┆ month_sin ┆ month_cos   ┆ day_of_year ┆ day_of_year ┆ wind_direct ┆ wind_direct ┆ … ┆ pressure_ms ┆ cloud_cover ┆ cloud_cove ┆ vapour_pre ┆ wind_speed ┆ wind_gusts ┆ rainfall │
│ ---      ┆ ---      ┆ ---       ┆ ---         ┆ _sin        ┆ _cos        ┆ ion_10m_sin ┆ ion_10m_cos ┆   ┆ l           ┆ _low        ┆ r          ┆ ssure_defi ┆ _10m       ┆ _10m       ┆ ---      │
│ f64      ┆ f64      ┆ f64       ┆ f64         ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆   ┆ ---         ┆ ---         ┆ ---        ┆ cit        ┆ ---        ┆ ---        ┆ i8       │
│          ┆          ┆           ┆             ┆ f64         ┆ f64         ┆ f64         ┆ f64         ┆   ┆ f64         ┆ i8          ┆ i8         ┆ ---        ┆ f64        ┆ f64 

### Finalized ML Pipeline.

In [261]:
yclical_features = ["hour", "month", "day_of_year", "wind_direction_10m"]
cyclical_transformer = CyclicalFeatures(drop_original = True,
                                        max_values = {'hour':24, 'month':12, 'day_of_year':366, 'wind_direction_10m':360})

preprocessor: ColumnTransformer = ColumnTransformer( # below we've performed cyclic_cossin_transformer only.
    transformers = [("Cyclical_Transformer", cyclical_transformer, cyclical_features),], # (name, function, features_list).
    remainder = "passthrough", # remove the columns used above to trasform, keep the rest as they are after newly transformed 6 columns..
    verbose_feature_names_out = False # Prefix the input features name (here it's e.g. hour_sin after function_transformer overridden 
)                                     # the original input feature name) with their corresponding transformer_name or not.

xgbClf_pipeline = Pipeline(steps = [("preprocessor", preprocessor),
                                    ("classifier",   XGBClassifier(objective = 'multi:softmax',
                                                                   num_class = 5,
                                                                   random_state = 42))])

### Splitting Dataset.

In [263]:
X = df_pandas.drop(columns = 'rainfall') # 2D Shape.
y = df_pandas['rainfall'] # 1D Shape.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

### Finding besting Hyperparameters with RandomizedSearchCV.

In [264]:
# Refined parameter grid
parameters = {
    'classifier__n_estimators'      : [100, 200, 300],
    'classifier__learning_rate'     : [0.01, 0.05, 0.1, 0.2],
    'classifier__max_depth'         : [3, 4, 5, 6, 7],
    'classifier__subsample'         : [0.8, 0.9, 1.0],
    'classifier__colsample_bytree'  : [0.3, 0.5, 0.7, 0.9],
    'classifier__gamma'             : [0, 0.1, 0.2],
    'classifier__reg_alpha'         : [0, 0.1, 1],
    'classifier__reg_lambda'        : [1, 1.5, 2],
} # 3 * 4 * 5 * 3 * 4 * 3 * 3 * 3 = 19440 combinations.

search = RandomizedSearchCV(estimator           = xgbClf_pipeline,
                            param_distributions = parameters,
                            n_iter              = 100,         # Out of 19440 combinations, only random 100 will be run.
                            cv                  = 5,           # 5 Fold Cross Validation.
                            scoring             = 'f1_macro',  # Robust metric for imbalance.
                            n_jobs              = -1,
                            random_state        = 42,          # For reproducible sampling
                            error_score         = 'raise')     # if an error occurs in estimator fitting, raise Errors!

search.fit(X = X_train, y = y_train)

# Comprehensive analysis
print(f"Best F1 Score = {search.best_score_}.")
print(f"\nBest Hyperparameters values : \n{search.best_params_}.")

best_pipeline = search.best_estimator_ # RandomSearchCV already trained this best model with training dataset.

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Best F1 Score = 0.48099244233991945.

Best Hyperparameters values : 
{'classifier__subsample': 0.8, 'classifier__reg_lambda': 2, 'classifier__reg_alpha': 0.1, 'classifier__n_estimators': 200, 'classifier__max_depth': 7, 'classifier__learning_rate': 0.2, 'classifier__gamma': 0.2, 'classifier__colsample_bytree': 0.7}.


In [269]:
accuracy_score(y_true = y_test, y_pred = best_pipeline.predict(X_test))

joblib.dump(value = best_pipeline, filename = r"D:\VS CODE\ML Projects\2_weather_prediction\weather_app\app\trained_models\weather_model.pkl")

['D:\\VS CODE\\ML Projects\\2_weather_prediction\\weather_app\\app\\trained_models\\weather_model.pkl']

In [ ]:
import plotly.express as px
import numpy as np
import pandas as pd

degrees = np.arange(0, 361)
radians = np.radians(degrees)
sinx = np.sin(radians)
cosx = np.cos(radians)

df = pd.DataFrame({
    "Degrees": degrees,
    "sin(x)": sinx,
    "cos(x)": cosx
})

fig = px.line(df, x="Degrees", y=["sin(x)", "cos(x)"], title="Sine & Cosine Waves")
fig.show()

In [ ]:
import plotly.express as px
import numpy as np
import pandas as pd

degrees = np.arange(0, 361)
radians = np.radians(degrees)
sinx = np.sin(radians)
cosx = np.cos(radians)

df = pd.DataFrame({
    "Degrees": degrees,
    "archtan2(x)": np.arctan2(sinx, cosx)
})

fig = px.line(df, x="Degrees", y="archtan2(x)", title="Sine & Cosine Waves")
fig.show()

In [ ]:
import numpy as np
import pandas as pd
import polars as pl

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import FunctionTransformer

df = fetch_openml("Bike_Sharing_Demand", version=2, as_frame=True).frame

# df = pl.DataFrame(df)
print(df.head())

   season  year  month  hour holiday  weekday workingday weather  temp  \
0  spring     0      1     0   False        6      False   clear  9.84   
1  spring     0      1     1   False        6      False   clear  9.02   
2  spring     0      1     2   False        6      False   clear  9.02   
3  spring     0      1     3   False        6      False   clear  9.84   
4  spring     0      1     4   False        6      False   clear  9.84   

   feel_temp  humidity  windspeed  count  
0     14.395      0.81        0.0     16  
1     13.635      0.80        0.0     40  
2     13.635      0.80        0.0     32  
3     14.395      0.75        0.0     13  
4     14.395      0.75        0.0      1  


In [ ]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

OrdinalEncoder()

cat_features = ['weather']
oe = OrdinalEncoder(handle_unknown = "use_encoded_value", unknown_value = -1)

preprocessor = ColumnTransformer(transformers = [("categorical", oe, cat_features)],
verbose_feature_names_out = lambda transformers_names, features_names: f"{features_names}_t").set_output(transform="pandas")

preprocessor.fit_transform(df)
preprocessor.get_feature_names_out()

array(['weather_t'], dtype=object)

In [ ]:
def sin_transformer(period):
    return FunctionTransformer(
            func              = lambda degree: np.sin(degree * (2*np.pi / period)), # returns Radian Value.
            feature_names_out = lambda transformer_name, features_names: [f"{f}_sin" for f in features_names])

def cos_transformer(period):
    return FunctionTransformer(
            func              = lambda degree: np.cos(degree * (2*np.pi / period)), # returns Radian Value.
            feature_names_out = lambda transformer_name, features_names: [f"{f}_cos" for f in features_names])

preprocessor: ColumnTransformer = ColumnTransformer( # below we've performed cyclic_cossin_transformer only.
        transformers = [
            ("month_sin_transformer",   sin_transformer(12), ["month"]),
            ("month_cos_transformer",   cos_transformer(12), ["month"]),
            ("weekday_sin_transformer", sin_transformer(7), ["weekday"]),
            ("weekday_cos_transformer", cos_transformer(7), ["weekday"]),
            ("hour_sin_transformer",    sin_transformer(24), ["hour"]),
            ("hour_cos_transformer",    cos_transformer(24), ["hour"]),
        ],
    remainder = "passthrough", # remove the columns used above to trasform, keep the rest as they are after newly transformed 6 columns..
    verbose_feature_names_out = False # OR lambda transformers_names, features_names: f"{features_names}_t".
   
)

preprocessor.set_output(transform = "polars")

Xt = preprocessor.fit_transform(df)

print(Xt), preprocessor.get_feature_names_out()

          month_sin  weekday_cos  hour_cos  season  year holiday workingday  \
0      5.000000e-01      0.62349  1.000000  spring     0   False      False   
1      5.000000e-01      0.62349  0.965926  spring     0   False      False   
2      5.000000e-01      0.62349  0.866025  spring     0   False      False   
3      5.000000e-01      0.62349  0.707107  spring     0   False      False   
4      5.000000e-01      0.62349  0.500000  spring     0   False      False   
...             ...          ...       ...     ...   ...     ...        ...   
17374 -2.449294e-16      0.62349  0.258819  spring     1   False       True   
17375 -2.449294e-16      0.62349  0.500000  spring     1   False       True   
17376 -2.449294e-16      0.62349  0.707107  spring     1   False       True   
17377 -2.449294e-16      0.62349  0.866025  spring     1   False       True   
17378 -2.449294e-16      0.62349  0.965926  spring     1   False       True   

      weather   temp  feel_temp  humidity  windspee

(None,
 array(['month_sin', 'weekday_cos', 'hour_cos', 'season', 'year',
        'holiday', 'workingday', 'weather', 'temp', 'feel_temp',
        'humidity', 'windspeed', 'count'], dtype=object))